<a href="https://colab.research.google.com/github/desiredsoul-1513/foryou/blob/main/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### Install libtorrent and Initialize Session

In [1]:
# Cell combined with modern settings and speed optimizations
!pip uninstall -y lbry-libtorrent libtorrent python-libtorrent
!pip install libtorrent rich

import libtorrent as lt
import warnings

# Correctly suppress DeprecationWarnings for a cleaner output
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Configure optimized session settings for maximum speed
settings = {
    'listen_interfaces': '0.0.0.0:6881',
    'download_rate_limit': 0,
    'upload_rate_limit': 0,
    'active_downloads': 30,
    'active_seeds': 30,
    'active_limit': 60,
    'cache_size': 4096,
    'connections_limit': 200,
    'enable_dht': True,
    'enable_lsd': True,
    'enable_upnp': True,
    'enable_natpmp': True,
    'mixed_mode_algorithm': 0
}

ses = lt.session(settings)

# Adding DHT routers via the session method
routers = [
    ('router.bittorrent.com', 6881),
    ('router.utorrent.com', 6881),
    ('router.bitcomet.com', 6881)
]

for host, port in routers:
    ses.add_dht_router(host, port)

downloads = []

Found existing installation: libtorrent 2.0.11
Uninstalling libtorrent-2.0.11:
  Successfully uninstalled libtorrent-2.0.11
  Using cached libtorrent-2.0.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (362 bytes)
Using cached libtorrent-2.0.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.5 MB)


### Mount Google Drive
To stream files we need to mount Google Drive.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Add From Torrent File
You can run this cell to add more files as many times as you want

### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [3]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break

    try:
        # Using parse_magnet_uri and add_torrent to avoid DeprecationWarning
        atp = lt.parse_magnet_uri(magnet_link)
        atp.save_path = params["save_path"]
        downloads.append(ses.add_torrent(atp))
        print(f"Added: {magnet_link[:40]}...")
    except Exception as e:
        print(f"Error adding magnet link: {e}")

Enter Magnet Link Or Type Exit: magnet:?xt=urn:btih:a36631000e651c8b1130248e592c5937b43c72da&dn=www.1TamilMV.gratis%20-%2029%20%282026%29%20Tamil%20HQ%20PreDVD%20-%20x264%20-%20HQ%20Clean%20-%20AAC%20-%20700MB.mkv&xl=756925283&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337%2Fannounce&tr=udp%3A%2F%2Ftracker.torrent.eu.org%3A451%2Fannounce&tr=udp%3A%2F%2Ftracker.dler.org%3A6969%2Fannounce&tr=udp%3A%2F%2Fopen.stealth.si%3A80%2Fannounce&tr=udp%3A%2F%2Fopen.demonii.com%3A1337%2Fannounce&tr=https%3A%2F%2Ftracker.moeblog.cn%3A443%2Fannounce&tr=udp%3A%2F%2Fopen.dstud.io%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.srv00.com%3A6969%2Fannounce&tr=https%3A%2F%2Ftracker.zhuqiy.com%3A443%2Fannounce&tr=https%3A%2F%2Ftracker.pmman.tech%3A443%2Fannounce
Added: magnet:?xt=urn:btih:a36631000e651c8b1130...
Enter Magnet Link Or Type Exit: exit


### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [4]:
import time
from rich.progress import Progress, TextColumn, BarColumn, DownloadColumn, TransferSpeedColumn, TimeRemainingColumn, SpinnerColumn
from rich.live import Live
from rich.table import Table
from rich.panel import Panel
from rich.console import Console
from rich.layout import Layout
from rich import box

state_str = ["Queued", "Checking", "Metadata", "Downloading", "Finished", "Seeding", "Allocating", "Checking Resume"]
console = Console()

# Main progress display
job_progress = Progress(
    SpinnerColumn(),
    TextColumn("[bold cyan]{task.description}", justify="left"),
    BarColumn(bar_width=None, pulse_style="bold white"),
    "[progress.percentage]{task.percentage:>3.1f}%",
    "•",
    DownloadColumn(),
    "•",
    TransferSpeedColumn(),
    "•",
    TimeRemainingColumn(),
)

task_map = {}

def make_dashboard(prog, download_list):
    # Stats Table
    stats = Table.grid(expand=True)
    stats.add_column(justify="left", ratio=1)
    stats.add_column(justify="right", ratio=1)

    total_dl = 0
    total_ul = 0
    for d in download_list:
        s = d.status()
        total_dl += s.download_rate
        total_ul += s.upload_rate

    stats.add_row(
        f" [bold blue]Active Torrents:[/bold blue] {len(download_list)}",
        f"[bold green]Total DL:[/bold green] {total_dl / 1e6:.2f} MB/s  [bold red]Total UL:[/bold red] {total_ul / 1e6:.2f} MB/s "
    )

    # Layout assembly
    layout = Layout()
    layout.split_column(
        Layout(Panel(stats, title="[bold yellow]System Status", border_style="yellow", box=box.ROUNDED), size=3),
        Layout(Panel(prog, title="[bold green]Active Downloads", border_style="bright_blue", box=box.ROUNDED))
    )
    return layout

with Live(make_dashboard(job_progress, downloads), refresh_per_second=4) as live:
    while downloads:
        for download in downloads[:]:
            s = download.status()
            handle_id = id(download)

            name = s.name if s.has_metadata else f"[Fetching] {str(s.info_hash)[:10]}"

            if handle_id not in task_map:
                task_map[handle_id] = job_progress.add_task(description=name, total=100)

            job_progress.update(
                task_map[handle_id],
                description=name,
                completed=s.progress * 100,
            )

            if s.is_seeding or s.state == lt.torrent_status.finished:
                job_progress.remove_task(task_map[handle_id])
                del task_map[handle_id]
                ses.remove_torrent(download)
                downloads.remove(download)
                console.log(f"[bold green]✔ Completed:[/bold green] {name}")

        live.update(make_dashboard(job_progress, downloads))
        if not downloads:
            break
        time.sleep(0.5)

console.print("[bold green]All downloads finished successfully! [/bold green]🚀")

Output()

[18:38:56] ✔ Completed: www.1TamilMV.gratis - 29 (2026) Tamil HQ PreDVD - x264 - HQ Clean - AAC -  ]8;id=899795;file:///tmp/ipykernel_19350/2088930910.py\2088930910.py]8;;\:]8;id=154470;file:///tmp/ipykernel_19350/2088930910.py#77\77]8;;\
           700MB.mkv                                                                                               

All downloads finished successfully! 🚀